In [ ]:
import scipy
import numpy as np
import degirum as dg, mytools
from mAP_reid_utils import *

In [ ]:
cloud_token = mytools.get_token() # get cloud API access token from env.ini file
cloud_zoo_url = mytools.get_cloud_zoo_url() # get cloud zoo URL from env.ini file
# zoo = dg.connect(dg.CLOUD, cloud_zoo_url, cloud_token)
zoo = dg.connect_model_zoo("0.0.0.0:8778")
model = zoo.load_model("person_reidentification_retail--256x128_float_openvino_cpu_1")

## Prepare the Dataset

In [ ]:
!python3 prepare_dataset.py --dataset_path "../openvino_zoo/Market-1501-v15.09.15/market1501/raw/Market-1501-v15.09.15/"

## Extract the feature embeddings using the reid model

In [ ]:
query_rootdir = "../openvino_zoo/Market-1501-v15.09.15/market1501/raw/Market-1501-v15.09.15/pytorch/gallery/"
gallery_rootdir = "../openvino_zoo/Market-1501-v15.09.15/market1501/raw/Market-1501-v15.09.15/pytorch/gallery_features.json"
query_file_path = "../openvino_zoo/Market-1501-v15.09.15/market1501/raw/Market-1501-v15.09.15/pytorch/query_features.json"
gallery_file_path = "../openvino_zoo/Market-1501-v15.09.15/market1501/raw/Market-1501-v15.09.15/pytorch/gallery_features.json"

In [ ]:
import os
import json
count=0
query_features=[]
gallery_features=[]
for subdir, dirs, files in os.walk(query_rootdir):
    for file in files:
        count+=1
        query_features.append(model(os.path.join(subdir, file)).results[0]["data"])       
        
list_of_lists = [arr.tolist() for arr in query_features]

with open(gallery_file_path, "w") as json_file:
    json.dump(list_of_lists, json_file)

print(f"List saved to {query_file_path}")

count=0
features=[]
for subdir, dirs, files in os.walk(gallery_rootdir):
    for file in files:
        count+=1
        gallery_features.append(model(os.path.join(subdir, file)).results[0]["data"])      
        
list_of_lists = [arr.tolist() for arr in gallery_features]

with open(query_file_path, "w") as json_file:
    json.dump(list_of_lists, json_file)

print(f"List saved to {gallery_file_path}")

## Load the features,labels and cam for both query and gallery

In [ ]:
result = scipy.io.loadmat('./result.mat')
query_feature = result['query_f']
query_cam = result['query_cam'][0]
query_label = result['query_label'][0]
gallery_feature = result['gallery_f']
gallery_cam = result['gallery_cam'][0]
gallery_label = result['gallery_label'][0]

In [ ]:
query_feature.shape,query_label.shape,query_cam.shape,gallery_feature.shape,gallery_label.shape,gallery_cam.shape

## Computing mAP and rank@1 accuracy

In [ ]:
CMC = np.zeros(len(gallery_label), dtype=int)
ap = 0.0
for i in range(len(query_label)):
    ap_tmp,CMC_tmp = evaluate(query_feature[i],query_label[i],query_cam[i],gallery_feature,gallery_label,gallery_cam)
    if CMC_tmp[0]==-1:
        continue
    CMC = CMC + CMC_tmp
    ap += ap_tmp
CMC = CMC/len(query_label) #average CMC
print('Rank@1:%f Rank@5:%f Rank@10:%f mAP:%f'%(CMC[0],CMC[4],CMC[9],ap/len(query_label)))